# Agent extensions (GPU): no-progress stop (np1), feasibility step (fs1), vLLM backend

Runs `eval/runner_agent_ext.py` (a wrapper around the unchanged `eval/runner_agent.py`) on the agent scenarios.
Every extension is off unless set below; the system name gets the extension tags (`agent_full_np1_k3`,
`agent_full_fs1`, `..._vllm`), so nothing mixes with the existing `agent_full` / `agent_feedback` / `agent_naive` runs.

- **np1** `NO_PROGRESS_K` (default 3; 0 = off): stop with ESCALATE after K consecutive repair attempts without progress.
  With `max_attempts` = 3, K = 3 only turns a REJECT into an ESCALATE; K = 1 or 2 also saves calls.
  The offline replay (`eval/ext_replay_no_progress.py`, CPU) already gives the exact numbers for K = 1, 2, 3 on the
  recorded test runs; this notebook confirms them with real runs.
- **fs1** `FEASIBILITY = True`: one "can this be done?" call before the first repair (prompt a1-v4-fs1).
- **vllm** `BACKEND = "vllm"`: same runs on vLLM (its own version; outputs may differ from transformers).

After the run: download `agent_ext_bundle.zip`, unzip it at the project root (results land in
`results/stage2_agent/<system>/`), then `.venv/bin/python eval/agent_metrics.py results/stage2_agent test`.
Settings: GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. Resumable and cached.

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True
STAGE = "test"                 # "dev" for a quick check
NO_PROGRESS_K = 3              # np1: 0 = off
FEASIBILITY = False            # fs1
BACKEND = "hf"                 # "hf" | "vllm" | "reference_policy" (pipeline check only, NOT results)
VERIFIER = "oracle"
SYSTEMS = [("agent_full", "full", [])]       # e.g. add ("agent_feedback", "feedback", []) for np1 on a baseline
CATEGORIES = ["R", "C", "T", "E", "S"]
LIMIT = None
OUT = "/kaggle/working/results/stage2_agent"
CACHE = "/kaggle/working/llm_cache"
NEEDED = ["eval/runner_agent_ext.py", "src/igt/agents/repair_agent_ext.py", "eval/runner_agent.py"]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox / verifier / tools (own folder) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
if BACKEND == "vllm":
    subprocess.run(["pip", "install", "-q", "-r", "requirements-vllm.txt"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers; print('pandas', pandas.__version__, '| numpy', numpy.__version__, "
         "'| torch', torch.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
def sh(cmd, log=None):
    """Run a step, stream its output live, keep the full log."""
    t0 = time.time()
    p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {' '.join(cmd)}\n" + "".join(lines) + f"\n--- exit {rc}, {(time.time() - t0) / 60:.1f} min ---\n")
    assert rc == 0, f"failed: {' '.join(cmd)}"
    return (time.time() - t0) / 60

for system, mode, flags in SYSTEMS:
    cmd = ["python", "eval/runner_agent_ext.py", "--system", system, "--mode", mode, "--backend", BACKEND,
           "--model", MODEL_ID, "--verifier", VERIFIER, "--splits", STAGE, "--categories", *CATEGORIES,
           "--cache-dir", CACHE, "--out", OUT, "--stop-after-no-progress", str(NO_PROGRESS_K), *flags]
    cmd += ["--load-in-4bit"] if LOAD_IN_4BIT else []
    cmd += ["--feasibility-check"] if FEASIBILITY else []
    cmd += ["--limit", str(LIMIT)] if LIMIT else []
    print(f"== {system}: {(sh(cmd, f'{OUT}/logs/{system}_ext_{STAGE}.log')):.1f} min")

In [ ]:
# ---------------- Summary ----------------
p = subprocess.run(["python", "eval/agent_metrics.py", OUT, STAGE], env=ENV, capture_output=True, text=True)
print(p.stdout[-4000:], p.stderr[-1500:])
for d in sorted(os.listdir(OUT)):
    if os.path.isdir(f"{OUT}/{d}") and d != "logs":
        open(f"{OUT}/{d}/git_commit.txt", "w").write(COMMIT)
        if os.path.exists(f"{OUT}/{d}/run_info_ext.json"):
            print(d, json.load(open(f"{OUT}/{d}/run_info_ext.json"))["agent_version"])

In [ ]:
# ---------------- ONE zip with everything needed on the laptop (unzip at the project root) ----------------
BUNDLE = f"{WORK}/agent_ext_bundle.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["/kaggle/working/results/stage2_agent"]:
        for dp, _, fs in os.walk(base):
            for f in fs:
                full = os.path.join(dp, f)
                z.write(full, os.path.relpath(full, WORK))
    z.writestr("agent_ext_bundle_info.json", json.dumps(dict(commit=COMMIT, created=datetime.datetime.now().isoformat(timespec="seconds")), indent=2))
print("download:", BUNDLE, f"({os.path.getsize(BUNDLE) / 1e6:.1f} MB)")